# **Generative LLM: Llama 3**

* Notebook to generate speeches of german politicians
* Model used: german version of Llama 3, with 8 billion parameters $\to$ "DiscoResearch/Llama3-DiscoLeo-Instruct-8B-v0.1" [link to model](https://huggingface.co/DiscoResearch/Llama3-DiscoLeo-Instruct-8B-v0.1)
* Furthermore, tested model answers on political survey data ("Wahl-O-Mat")

In [1]:
!pip install datasets
!pip install torch
!pip install transformers -U
!pip install peft
!pip install bitsandbytes -U
!pip install -U trl
!pip install nltk

  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached pyarrow-21.0.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (3.3 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-3.5.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached hf_xet-1.1.7-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (703 bytes)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
Using cached datasets-4.0.0-py3-none-any.whl (494 kB)
Using cached huggingface_hub-0.34.4-py3-none-any.whl (561 kB)
Using cached multiprocess-0.70.16-py311-none-any.whl (143 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached p

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [3]:
import subprocess
import sys
from datasets import Dataset, load_dataset, DatasetDict
import re
import string
import seaborn as sns
import os
import pandas as pd
import random
import torch 
import numpy as np
from datasets import Dataset
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline
)
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report
import warnings
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
import json
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.empty_cache()

[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## **1) Import Speech Data**

* Clean Linebreaks
* Limit Speech Length
* Remove Greetings

In [5]:
data = pd.read_csv("../data/final_data.csv", on_bad_lines='skip')

# drop unnamed
if "Unnamed: 0" in data.columns:
    data = data.drop("Unnamed: 0", axis=1)

data["party"] = data["party"].replace("CDU/CSU", "Union")
data

,speech_text,legislative_period,protocol_nr,agenda_item_number,party,agenda_item_title,date,full_name
0,Herr Präsident! Kolleginnen und Kollegen! Die ...,19,10,3,Union,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Hans Michelbach
1,Sehr geehrter Herr Präsident! Liebe Kolleginne...,19,10,3,SPD,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Ingrid Arndt-Brauer
2,Herr Präsident! Liebe Kolleginnen und Kollegen...,19,10,3,Union,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Antje Tillmann
3,Herr Präsident! Liebe Kolleginnen und Kollegen...,19,10,3,GRÜNE,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Gerhard Schick
4,Liebe Kolleginnen und Kollegen! Es ist vorhin ...,19,10,3,FDP,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Florian Toncar
...,...,...,...,...,...,...,...,...
36112,Sehr geehrter Herr Präsident! Liebe Kolleginne...,20,99,5,SPD,Wolfsbestandsmanagement,2023-04-26,Lina Seitzl
36113,Frau Präsidentin! Meine Damen und Herren! Sie ...,20,99,3,GRÜNE,Aktuelle Stunde - Umstrittene Personalpolitik ...,2023-04-26,Till Steffen
36114,Sehr geehrter Herr Präsident! Meine Damen und ...,20,99,4,AfD,Bundeswehreinsatz Evakuierung aus Sudan,2023-04-26,Joachim Wundrak
36115,Sehr geehrte Präsidentin! Liebe Kolleginnen un...,20,99,7,LINKE,Unregulierte Massenmigration,2023-04-26,Clara Bünger


In [6]:
# clean line breaks and special spaces
data["cleaned_text"] = data["speech_text"].apply(lambda x: x.replace("\xa0", " "))
data["cleaned_text"] = data["cleaned_text"].apply(lambda x: x.replace("\n", " "))

# remove repeated spaces
data["cleaned_text"] = data["cleaned_text"].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

In [7]:
# extract count of words
data["word_count"] = data["cleaned_text"].apply(lambda x: len(x.split()))

# limit speech length
data_filtered = data[data["word_count"] >= 200][data["word_count"] < 1501]

print("Nr. of Training Speeches:", len(data_filtered))
data_filtered

Nr. of Training Speeches: 36117


,speech_text,legislative_period,protocol_nr,agenda_item_number,party,agenda_item_title,date,full_name,cleaned_text,word_count
0,Herr Präsident! Kolleginnen und Kollegen! Die ...,19,10,3,Union,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Hans Michelbach,Herr Präsident! Kolleginnen und Kollegen! Die ...,520
1,Sehr geehrter Herr Präsident! Liebe Kolleginne...,19,10,3,SPD,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Ingrid Arndt-Brauer,Sehr geehrter Herr Präsident! Liebe Kolleginne...,693
2,Herr Präsident! Liebe Kolleginnen und Kollegen...,19,10,3,Union,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Antje Tillmann,Herr Präsident! Liebe Kolleginnen und Kollegen...,710
3,Herr Präsident! Liebe Kolleginnen und Kollegen...,19,10,3,GRÜNE,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Gerhard Schick,Herr Präsident! Liebe Kolleginnen und Kollegen...,752
4,Liebe Kolleginnen und Kollegen! Es ist vorhin ...,19,10,3,FDP,Aktuelle Stunde zu einer europäischen Bankenunion,2018-01-31,Florian Toncar,Liebe Kolleginnen und Kollegen! Es ist vorhin ...,872
...,...,...,...,...,...,...,...,...,...,...
36112,Sehr geehrter Herr Präsident! Liebe Kolleginne...,20,99,5,SPD,Wolfsbestandsmanagement,2023-04-26,Lina Seitzl,Sehr geehrter Herr Präsident! Liebe Kolleginne...,837
36113,Frau Präsidentin! Meine Damen und Herren! Sie ...,20,99,3,GRÜNE,Aktuelle Stunde - Umstrittene Personalpolitik ...,2023-04-26,Till Steffen,Frau Präsidentin! Meine Damen und Herren! Sie ...,789
36114,Sehr geehrter Herr Präsident! Meine Damen und ...,20,99,4,AfD,Bundeswehreinsatz Evakuierung aus Sudan,2023-04-26,Joachim Wundrak,Sehr geehrter Herr Präsident! Meine Damen und ...,611
36115,Sehr geehrte Präsidentin! Liebe Kolleginnen un...,20,99,7,LINKE,Unregulierte Massenmigration,2023-04-26,Clara Bünger,Sehr geehrte Präsidentin! Liebe Kolleginnen un...,480


In [8]:
# list of greeting-related words
greeting_words = [
    "Damen", "Herren", "Herr", "Kollegen", "Kolleginnen", "Präsident", "Präsidentin",
    "Kollege", "Kollegin", "verehrte", "verehrten", "geehrte", "geehrter", "geehrten"
]

# Build a regex pattern to match any of these words as whole words (case insensitive)
greeting_pattern = re.compile(r'\b(?:' + '|'.join(greeting_words) + r')\b', flags=re.IGNORECASE)

def remove_greeting_sentences(text, max_sentences=10):
    # Tokenize into sentences
    sentences = sent_tokenize(text, language='german')
    
    cleaned_sentences = []
    for i, sentence in enumerate(sentences):
        if i < max_sentences and greeting_pattern.search(sentence):
            continue  # Skip greeting sentence
        cleaned_sentences.append(sentence)
    
    return ' '.join(cleaned_sentences)

# apply
data_filtered['cleaned_text'] = data_filtered['cleaned_text'].apply(remove_greeting_sentences)

# sainity check
print(data_filtered["cleaned_text"][100])

Als ich 2013 ganz frisch im Bundestag die Berichterstattung für digitale Bildung übernommen habe, wusste mit diesen Begriffen noch kaum jemand etwas anzufangen, außer natürlich einer eingeschworenen Gemeinschaft digitalaffiner Lehrkräfte im Twitterlehrerzimmer. Seither sind wir wesentlich weitergekommen. Das freut mich ungemein. Dann kam die Ankündigung der damaligen Bildungsministerin Wanka eines DigitalPakts in der „BamS“. Jetzt haben wir den DigitalPakt, und zwar mit Brief und Siegel. Das ist großartig. Das ist ein Anlass zur Freude, zum Durchatmen, aber bitte nicht zu lang. Wir leben in einer Welt, die niemals stehen bleibt, in der digitale Medien, die Methode digitalen Lernens und Arbeitens immer selbstverständlicher werden. Die Schule muss dazu ermutigen und befähigen. Insofern bin ich der Opposition dankbar, als sie den DigitalPakt weiterdenkt und weiterentwickelt. Das steht heute nicht an, aber es muss weitergehen, ganz klar. Auch ich bin überzeugt: Der DigitalPakt muss dauerha

## **2) Sample Speeches**

* Ensure same number of training speeches per party

In [9]:
parties_count = data_filtered[["cleaned_text", "party"]].groupby("party").count().reset_index()
parties_count

,party,cleaned_text
0,AfD,5041
1,FDP,4683
2,GRÜNE,5067
3,LINKE,3618
4,SPD,8032
5,Union,9676


In [10]:
# As we agreed to a balance sample, I shorten the dataframe to te maximmum length of LINKE

max_length = parties_count["cleaned_text"].min() #3618
parties = data_filtered["party"].unique()

# final df
final_df = pd.DataFrame()


for party in parties:
    # party df
    party_unique_df = data_filtered[data_filtered["party"] == party]

    if len(party_unique_df) >= max_length: # if too many speeches, sample
        party_sample = party_unique_df.sample(n=max_length, random_state=42)
    else:
        party_sample = party_unique_df


    final_df = pd.concat([final_df, party_sample], ignore_index=True)


final_df.head()

,speech_text,legislative_period,protocol_nr,agenda_item_number,party,agenda_item_title,date,full_name,cleaned_text,word_count
0,Sehr geehrte Frau Präsidentin! Werte Kolleginn...,20,79,12,Union,EU-Richtlinie Umweltauswirkungen Kunststoffpro...,2023-01-19,Björn Simon,Auch von meiner Seite aus einen herzlichen Glü...,789
1,Hochgeschätzter Herr Präsident! Kolleginnen un...,19,23,6,Union,Verkehr und digitale Infrastruktur,2018-03-22,Andreas Scheuer,Luftqualität ist Lebensqualität; aber Lebensqu...,1292
2,Frau Präsidentin! Verehrte Kolleginnen und Kol...,20,18,8,Union,Aktuelle Stunde - Bundeswehreinsätze in Mali b...,2022-02-18,Reinhard Brandl,Die Ampelkoalition kann jetzt nichts für die S...,741
3,Frau Präsidentin! Liebe Kolleginnen und Kolleg...,20,21,4,Union,Meinungsfreiheit in Sozialen Medien,2022-03-17,Marc Henrichmann,Das vierte Wort im Antrag der AfD ist „Russlan...,936
4,Herr Präsident! Meine sehr verehrten Damen und...,19,164,3,Union,Bundeswehreinsatz EUTM Mali,2020-05-29,Johann David Wadephul,Die Lage im Sahel ist kritisch. Sie ist sogar ...,1011


In [13]:
system_prompt = """Du bist ein*e Abgeordnete*r des Deutschen Bundestages. 
Gegeben einer Partei und eines Themas schreibst du eine deutsche, 
politische Rede mit bis zu 800 Wörtern."""

def make_prompt(row):
    return {
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": f"Schreibe eine politische Rede der {row['party']} zum Thema {row['agenda_item_title']}."
            },
            {
                "role": "assistant",
                "content": row["cleaned_text"]
            }
        ]
    }

# create the prompt column
final_df["prompt"] = final_df.apply(make_prompt, axis=1)

# save to JSONL
with open("../data/training_speeches.jsonl", "w", encoding="utf-8") as f:
    for example in final_df["prompt"]:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

### **Topic Definition**

In [14]:
# Dictionary of party and topics
topic_to_speech = {"Union" : ["Mindestlohn", "Bundeswehreinsatz im Kosovo", "Wirtschaftshilfen Corona", "Gaspreise"],
                   "SPD" : ["Mindestlohn", "Bundeswehreinsatz im Kosovo", "Wirtschaftshilfen Corona", "Gaspreise"],
                   "GRÜNE" : ["Mindestlohn", "Bundeswehreinsatz im Kosovo", "Wirtschaftshilfen Corona", "Gaspreise"],
                   "FDP" : ["Mindestlohn", "Bundeswehreinsatz im Kosovo", "Wirtschaftshilfen Corona", "Gaspreise"],
                   "AfD" : ["Mindestlohn", "Bundeswehreinsatz im Kosovo", "Wirtschaftshilfen Corona", "Gaspreise"],
                   "Linke" : ["Mindestlohn", "Bundeswehreinsatz im Kosovo", "Wirtschaftshilfen Corona", "Gaspreise"]}

## **3.) Import and Run the Base Model**

In [12]:
device="cuda"

base_model = AutoModelForCausalLM.from_pretrained(
    "DiscoResearch/Llama3-DiscoLeo-Instruct-8B-v0.1",
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("DiscoResearch/Llama3-DiscoLeo-Instruct-8B-v0.1")
tokenizer.pad_token = tokenizer.eos_token  # Llama has no pad token

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [15]:
def dynamic_max_new_tokens(prompt_text, max_context=2048, safety_margin=50, words_target = None):
    prompt_len = len(tokenizer(prompt_text)["input_ids"])
    available = max_context - prompt_len - safety_margin
    if words_target:
        target_tokens = int(words_target * TOKENS_PER_WORD)
        return min(available, target_tokens)
    return available


def generate_speech(party, topic, system_prompt, model, tokenizer, temp, device="cuda"):
    """
    Generate a political speech for a given party and topic.
    """
    prompt = f"Schreibe eine politische Rede der {party} zum Thema {topic}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    # Prepare model input
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt", padding=True, truncation=True, max_length=2048).to(device)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Generate output
    max_new_tokens = dynamic_max_new_tokens(text) # max left tokens after prompt tokenization
    
    generated_ids = model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,  
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temp,
        top_p=0.9,
        repetition_penalty=1.2, # penalize the model for repetitions of previously generated tokens
        pad_token_id=tokenizer.pad_token_id 
    )
    
    # Remove prompt tokens from output
    generated_ids = [
        output_ids[len(input_ids):] 
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    # Decode to text
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return response



def generate_all_speeches(topic_dict, system_prompt, model, tokenizer, temp, device="cuda"):
    """
    Generate speeches for all (party, topic) pairs in topic_dict
    and return a DataFrame with columns: party, topic, speech.
    """
    rows = []
    
    for party, topics in tqdm(topic_dict.items(), desc="Generating speeches"):
        for topic in topics:
            speech = generate_speech(party, topic, system_prompt, model, tokenizer, temp, device)
            rows.append({
                "party": party,
                "topic": topic,
                "speech": speech
            })
    
    df = pd.DataFrame(rows)
    return df



In [16]:
system_prompt = """Du bist ein*e Abgeordnete*r des Deutschen Bundestages. 
Gegeben einer Partei und eines Themas schreibst du eine deutsche, 
politische Rede mit bis zu 800 Wörtern."""


temperatures = {"03": 0.3, "05": 0.5, "07": 0.7}
all_speeches_df = None

# apply to base model

for label, temp in temperatures.items():
    speeches_df = generate_all_speeches(topic_to_speech, system_prompt, base_model, tokenizer, temp=temp)
    speeches_df = speeches_df.rename(columns={"speech": f"base_speech_{label}"})
    
    speeches_df.to_csv(f"../data/Llama_3_base_speeches_temp_{label}.csv", index=False)
    
    if all_speeches_df is None:
        all_speeches_df = speeches_df
    else:
        all_speeches_df = all_speeches_df.merge(speeches_df, on=["party", "topic"], how="inner")

# Save merged file
all_speeches_df.to_csv("../data/Llama_3_base_all_temps_speeches.csv", index=False)


Generating speeches: 100%|██████████| 6/6 [03:43<00:00, 37.25s/it]


## **4) Fine-tune the model**

In [12]:
# model name
model_name = "DiscoResearch/Llama3-DiscoLeo-Instruct-8B-v0.1"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Llama has no pad token

In [13]:
# load dataset
dataset = load_dataset("json", data_files="../data/training_speeches.jsonl")

Generating train split: 0 examples [00:00, ? examples/s]

In [14]:
def tokenize_chat(example):
    # example["messages"] is already a list of dicts
    full_text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=2048, # use full potential token length for training --> show the model as much speech as possible
        padding="max_length"
    )

    # Labels = copy of input_ids
    labels = tokenized["input_ids"].copy()

    # Mask loss before assistant's answer
    assistant_text = example["messages"][-1]["content"]
    assistant_token_ids = tokenizer(
        assistant_text,
        truncation=True,
        max_length=2048,
        padding=False
    )["input_ids"]

    for start_idx in range(len(labels) - len(assistant_token_ids) + 1):
        if labels[start_idx:start_idx + len(assistant_token_ids)] == assistant_token_ids:
            break

    labels[:start_idx] = [-100] * start_idx
    tokenized["labels"] = labels

    return tokenized


tokenized_dataset = dataset.map(tokenize_chat, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/21708 [00:00<?, ? examples/s]

In [15]:
# Load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [16]:
# empty cache before training
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# Training args
training_args = TrainingArguments(
    output_dir="./llama3-political-speeches-final",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5, 
    logging_steps=50,
    save_steps=500,
    num_train_epochs=3,
    bf16=True,
    optim="adamw_torch",
    eval_strategy="no"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"], #tokenized_dataset_sample
    tokenizer=tokenizer
)

trainer.train()
model.save_pretrained("./llama3-political-speeches-final")


Step,Training Loss
50,1.662100
100,0.691300
150,0.610500
200,0.589500
250,0.620100
300,0.517000
350,0.603700
400,0.599100
450,0.601100
500,0.613100


## **5) Generate Speeches with Fine-tuned Model**

In [18]:
def dynamic_max_new_tokens(prompt_text, max_context=2048, safety_margin=50, words_target = None):
    prompt_len = len(tokenizer(prompt_text)["input_ids"])
    available = max_context - prompt_len - safety_margin
    if words_target:
        target_tokens = int(words_target * TOKENS_PER_WORD)
        return min(available, target_tokens)
    return available



def generate_speech(party, topic, system_prompt, model, tokenizer, temp, device="cuda"):
    """
    Generate a political speech for a given party and topic.
    """
    prompt = f"Schreibe eine politische Rede der {party} zum Thema {topic}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    # Prepare model input
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt", padding=True, truncation=True, max_length=2048).to(device)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Generate output
    max_new_tokens = dynamic_max_new_tokens(text) # max left tokens after prompt tokenization
    
    generated_ids = model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,  
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temp,
        top_p=0.9,
        repetition_penalty=1.2, # penalize the model for repetitions of previously generated tokens
        pad_token_id=tokenizer.pad_token_id 
    )
    
    # Remove prompt tokens from output
    generated_ids = [
        output_ids[len(input_ids):] 
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    # Decode to text
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return response



def generate_all_speeches(topic_dict, system_prompt, model, tokenizer, temp, device="cuda"):
    """
    Generate speeches for all (party, topic) pairs in topic_dict
    and return a DataFrame with columns: party, topic, speech.
    """
    rows = []
    
    for party, topics in tqdm(topic_dict.items(), desc="Generating speeches"):
        for topic in topics:
            speech = generate_speech(party, topic, system_prompt, model, tokenizer, temp, device)
            rows.append({
                "party": party,
                "topic": topic,
                "speech": speech
            })
    
    df = pd.DataFrame(rows)
    return df





In [19]:
system_prompt = """Du bist ein*e Abgeordnete*r des Deutschen Bundestages. 
Gegeben einer Partei und eines Themas schreibst du eine deutsche, 
politische Rede mit bis zu 800 Wörtern."""



temperatures = {"03": 0.3, "05": 0.5, "07": 0.7}
all_speeches_df = None

for label, temp in temperatures.items():
    speeches_df = generate_all_speeches(topic_to_speech, system_prompt, model, tokenizer, temp=temp)
    speeches_df = speeches_df.rename(columns={"speech": f"finetuned_speech_{label}"})
    
    speeches_df.to_csv(f"../data/Llama_3_finetuned_speeches_temp_{label}.csv", index=False)
    
    if all_speeches_df is None:
        all_speeches_df = speeches_df
    else:
        all_speeches_df = all_speeches_df.merge(speeches_df, on=["party", "topic"], how="inner")

# Save merged file
all_speeches_df.to_csv("../data/Llama_3_finetuned_all_temps_speeches.csv", index=False)


Generating speeches: 100%|██████████| 6/6 [27:48<00:00, 278.03s/it]


In [22]:
# merge and save
base_all_speeches_df = pd.read_csv("../data/Llama_3_base_all_temps_speeches.csv")
finetuned_all_speeches_df = pd.read_csv("../data/Llama_3_finetuned_all_temps_speeches.csv")
final_df = base_all_speeches_df.merge(finetuned_all_speeches_df, on=["party", "topic"], how="inner")

final_df.to_csv("../data/Llama_3_speeches.csv", index = False)


## **6) Validate with Wahl-O-Mat**

In [20]:
# import
wahl_o_mat_data = pd.read_csv("../data/Wahl-O-Mat-Bundestagswahl-2025.csv", sep = ";")
wahl_o_mat_data

parties = ["SPD", "Union", "LINKE", "AfD", "GRÜNE", "FDP"]
# rename parties to match the names, the model learned
wahl_o_mat_data["Partei: Kurzbezeichnung"] = wahl_o_mat_data["Partei: Kurzbezeichnung"].replace("CDU / CSU", "Union")
wahl_o_mat_data["Partei: Kurzbezeichnung"] = wahl_o_mat_data["Partei: Kurzbezeichnung"].replace("Die Linke", "LINKE")

# restrict to desired parties and columns
wahl_o_mat_data = wahl_o_mat_data[wahl_o_mat_data["Partei: Kurzbezeichnung"].isin(parties)]
wahl_o_mat_data = wahl_o_mat_data[["Partei: Kurzbezeichnung", "These: These", "Position: Position"]].rename(columns = {"Partei: Kurzbezeichnung": "party",
                                                                                                                       "These: These" : "question",
                                                                                                                       "Position: Position" :"answer"})
wahl_o_mat_data

,party,question,answer
0,SPD,Deutschland soll die Ukraine weiterhin militär...,stimme zu
1,Union,Deutschland soll die Ukraine weiterhin militär...,stimme zu
2,GRÜNE,Deutschland soll die Ukraine weiterhin militär...,stimme zu
3,FDP,Deutschland soll die Ukraine weiterhin militär...,stimme zu
4,AfD,Deutschland soll die Ukraine weiterhin militär...,stimme nicht zu
...,...,...,...
1037,Union,Der gesetzliche Mindestlohn soll spätestens 20...,neutral
1038,GRÜNE,Der gesetzliche Mindestlohn soll spätestens 20...,stimme zu
1039,FDP,Der gesetzliche Mindestlohn soll spätestens 20...,stimme nicht zu
1040,AfD,Der gesetzliche Mindestlohn soll spätestens 20...,neutral


In [21]:
system_prompt_survey = """Du bist ein*e Abgeordnete*r des Deutschen Bundestages. 
Gegeben einer Partei und der damit verbundenen Standpunkte, wie würdest du auf die folgende Frage antworten? 
Antworte ausschließlich mit einer der folgenden Optionen: 'stimme zu', 'stimme nicht zu', 'neutral'"""


def generate_answer(party, question, system_prompt_survey, model, tokenizer, temp, device="cuda"):
    """
    Generate a survey answer for a given party and question.
    """
    prompt = f"Was antwortest du als Abgeordnete*r der Partei {party} auf die Frame {question}? Antworte ausschließlich mit einer der folgenden Optionen: 'stimme zu', 'stimme nicht zu', 'neutral'."
    
    messages = [
        {"role": "system", "content": system_prompt_survey},
        {"role": "user", "content": prompt}
    ]
    
    # Prepare model input
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt", padding=True, truncation=True, max_length=2048).to(device)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Generate output
    max_new_tokens = dynamic_max_new_tokens(text) # max left tokens after prompt tokenization
    
    generated_ids = model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,  
        max_new_tokens=20, # model only needs to answer 
        do_sample=True,
        temperature=temp,
        top_p=0.9,
        repetition_penalty=1.2, # penalize the model for repetitions of previously generated tokens
        pad_token_id=tokenizer.pad_token_id 
    )
    
    # Remove prompt tokens from output
    generated_ids = [
        output_ids[len(input_ids):] 
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    # Decode to text
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return response



In [22]:
temperatures = {"03": 0.3, "05": 0.5, "07": 0.7}

wahl_o_mat_results = wahl_o_mat_data.copy()

for label, temp in temperatures.items():
    answers = []
    for _, row in tqdm(wahl_o_mat_data.iterrows(), total=len(wahl_o_mat_data), desc=f"Temp {temp}"):
        ans = generate_answer(
            row["party"], 
            row["question"], 
            system_prompt_survey, 
            model, 
            tokenizer, 
            temp=temp
        )
        answers.append(ans)
    wahl_o_mat_results[f"answer_{label}"] = answers

# Save combined dataframe
wahl_o_mat_results.to_csv("../data/Llama_3_wahl_o_mat_answers.csv", index=False)
wahl_o_mat_results

Temp 0.7: 100%|██████████| 228/228 [01:19<00:00,  2.86it/s]


,party,question,answer,answer_03,answer_05,answer_07
0,SPD,Deutschland soll die Ukraine weiterhin militär...,stimme zu,stimme zu,stimme zu,stimme zu
1,Union,Deutschland soll die Ukraine weiterhin militär...,stimme zu,stimme zu,stimmte zu,stimme zu
2,GRÜNE,Deutschland soll die Ukraine weiterhin militär...,stimme zu,stimme zu,stimme zu,stimmte zu
3,FDP,Deutschland soll die Ukraine weiterhin militär...,stimme zu,stimmte zu,stimme zu,stimme zu
4,AfD,Deutschland soll die Ukraine weiterhin militär...,stimme nicht zu,stimmte zu,stimmte zu,stimme nicht zu
...,...,...,...,...,...,...
1037,Union,Der gesetzliche Mindestlohn soll spätestens 20...,neutral,stimmte zu,stimmte zu,stimmte zu
1038,GRÜNE,Der gesetzliche Mindestlohn soll spätestens 20...,stimme zu,stimmte zu,stimmte zu,stimme zu
1039,FDP,Der gesetzliche Mindestlohn soll spätestens 20...,stimme nicht zu,stimmte zu,stimme nicht zu,stimme nicht zu
1040,AfD,Der gesetzliche Mindestlohn soll spätestens 20...,neutral,stimmte zu,stimme nicht zu,stimmte zu
